# 5 — Chunking

We have raw text from PDFs. Now we need to split it into **chunks** — smaller pieces that we can embed and search over.

This sounds simple but it matters a lot. Bad chunks → bad retrieval → bad answers.

We'll try two approaches:
1. Naive fixed-size chunks
2. Paragraph-aware chunks

In [ ]:
with open("sample_text.txt", "r") as f:
    text = f.read()

print(f"Total characters: {len(text)}")
print(f"Preview: {text[:200]}...")

## Approach 1: Fixed-size chunks

The simplest strategy — cut the text every N characters. Add some overlap so we don't lose context at the boundaries.

In [ ]:
def chunk_fixed(text, chunk_size=500, overlap=50):
    chunks = []
    for i in range(0, len(text), chunk_size - overlap):
        chunks.append(text[i : i + chunk_size])
    return chunks

In [ ]:
naive_chunks = chunk_fixed(text)

print(f"Number of chunks: {len(naive_chunks)}")
print(f"\n--- Chunk 1 ---")
print(naive_chunks[0])

Now look at the boundary between two chunks:

In [ ]:
print("--- End of chunk 1 ---")
print(f"...{naive_chunks[0][-80:]}")
print()
print("--- Start of chunk 2 ---")
print(f"{naive_chunks[1][:80]}...")

### Problems with this approach

- Cuts words in half
- Cuts sentences in half
- A chunk can start mid-paragraph about one topic and end mid-paragraph about another
- The overlap helps a bit, but it's a band-aid

If the LLM gets a chunk that says `"...eans were so valuable in Mesoamerican cultures that th"` — that's not useful context.

## Approach 2: Split by paragraphs

A smarter approach — respect the natural structure of the text. Split on paragraph breaks (`\n\n`), then group small paragraphs together up to a max size.

This way each chunk contains complete thoughts.

In [ ]:
def chunk_by_paragraphs(text, max_chunk_size=1000):
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
    chunks = []
    current_chunk = ""
    for para in paragraphs:
        if len(current_chunk) + len(para) > max_chunk_size and current_chunk:
            chunks.append(current_chunk.strip())
            current_chunk = ""
        current_chunk += para + "\n\n"
    if current_chunk.strip():
        chunks.append(current_chunk.strip())
    return chunks

In [ ]:
smart_chunks = chunk_by_paragraphs(text)

print(f"Number of chunks: {len(smart_chunks)}")
for i, chunk in enumerate(smart_chunks):
    print(f"\n--- Chunk {i+1} ({len(chunk)} chars) ---")
    print(chunk[:200] + "...")

Much better — each chunk covers a coherent topic and contains complete sentences.

Let's look at one full chunk:

In [ ]:
print(smart_chunks[0])

## What about `max_chunk_size`?

This is a trade-off:

| Smaller chunks | Larger chunks |
|---|---|
| More precise retrieval | More context per chunk |
| Less noise in the prompt | Fewer API calls to embed |
| Risk of losing context | Risk of mixing unrelated topics |

There's no perfect answer — it depends on the documents and the questions. Experiment.

In [ ]:
for size in [500, 1000, 2000]:
    chunks = chunk_by_paragraphs(text, max_chunk_size=size)
    avg_len = sum(len(c) for c in chunks) / len(chunks)
    print(f"max_chunk_size={size:>5} → {len(chunks):>2} chunks, avg {avg_len:.0f} chars")